<!--
---
PURPOSE: Scaffold pose project and export frame samples.
REQUIRES:
  - outputs/video/video_assets.parquet
  - outputs/video/frame_times.parquet
PRODUCES:
  - pose_projects/session_{id}/{sleap|dlc}/...
  - outputs/labeling/sleap/{session_id}/{camera}/labels.csv
  - outputs/labeling/sleap/{session_id}/{camera}/frames/*.png
WHAT_NEXT: notebooks/07_Pose_to_Motifs_Feature_Engineering.ipynb
---
-->

# 06 Pose Estimation Setup (SLEAP or DLC)

**Purpose**
Scaffold pose project and export frame samples.

**Requires**
- `outputs/video/video_assets.parquet`
- `outputs/video/frame_times.parquet`

**Produces**
- `pose_projects/session_{id}/{sleap|dlc}/...`
- `outputs/labeling/sleap/{session_id}/{camera}/labels.csv`
- `outputs/labeling/sleap/{session_id}/{camera}/frames/*.png`

**What to run next**
- `notebooks/07_Pose_to_Motifs_Feature_Engineering.ipynb`

We scaffold a pose project and export frame samples with timestamps.


## Environment Setup
We add the repo `src/` to the Python path so notebooks can import shared modules.

In [7]:
import sys
import importlib
import inspect
from pathlib import Path

# Resolve repo root whether notebook was launched from repo root or notebooks/.
ROOT = Path.cwd().resolve()
if (ROOT / "src").exists() and (ROOT / "notebooks").exists():
    pass
elif (ROOT.parent / "src").exists() and (ROOT.parent / "notebooks").exists():
    ROOT = ROOT.parent
else:
    raise RuntimeError(f"Could not resolve project root from cwd={Path.cwd()}")

SRC = ROOT / "src"

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# Ensure notebook uses the local io_sessions module from this repo.
io_sessions = importlib.import_module("io_sessions")
io_sessions = importlib.reload(io_sessions)
print("io_sessions:", io_sessions.__file__)
print("get_session_bundle:", inspect.signature(io_sessions.get_session_bundle))


io_sessions: /Users/muh/projects/vbn-analysis/src/io_sessions.py
get_session_bundle: (session_id: 'int', sessions_df: 'pd.DataFrame | None' = None, *, resolve_nwb: 'bool' = True, inspect_modalities: 'bool' = True) -> 'SessionBundle'


## Prerequisite Check
We parse the notebook header and validate required artifacts before running downstream steps.

In [11]:
from pathlib import Path
from reports import parse_notebook_header, validate_prerequisites

nb_path = ROOT / "notebooks" / "06_Pose_Estimation_Setup_SLEAP_or_DLC.ipynb"
header  = parse_notebook_header(nb_path)
missing = validate_prerequisites(header.get("REQUIRES", []))

if missing:
    print("Missing prerequisites:")
    for item in missing:
        print(" -", item)
else:
    print("All prerequisites satisfied.")

All prerequisites satisfied.


/opt/anaconda3/envs/vbn-analysis/lib/python3.10/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


## Step 1: Scaffold pose project
This creates the directory structure for your chosen tool.

In [13]:
# Labeling parameters
LABELING_MODE = "mp4"  # "mp4" or "png"
N_SAMPLES = 50
LABEL_FPS = 30
CLIP_START_S = None  # e.g., 120.0
CLIP_DURATION_S = None  # e.g., 5.0
WRITE_PNGS = False  # only used for mp4 mode


In [15]:
import json
from pathlib import Path
from io_sessions import load_sessions_csv, get_session_bundle
from config import get_config
import importlib
import features_pose
importlib.reload(features_pose)
from features_pose import (
    sample_frame_indices,
    scaffold_pose_project,
    export_labeling_frames,
    export_labeling_video,
)

cfg = get_config()
sessions = load_sessions_csv()
SESSION_IDS = sessions.session_id.tolist()[:1]

for session_id in SESSION_IDS:
    bundle = get_session_bundle(session_id, sessions, resolve_nwb=False, inspect_modalities=False)
    assets = bundle.load_video_assets()
    frame_times = bundle.load_frame_times()
    if assets is None or assets.empty or frame_times is None or frame_times.empty:
        print("No video assets or frame times available.")
        continue

    project_dir = scaffold_pose_project(session_id, cfg.pose_tool, cfg.pose_projects_dir)
    for camera in assets["camera"].unique():
        row = assets[assets["camera"] == camera].iloc[0]
        video_path = row.get("local_video_path")
        if not video_path:
            print(f"No local video for camera: {camera}")
            continue
        ft_cam = frame_times[frame_times["camera"] == camera]
        if ft_cam.empty:
            print(f"No frame times for camera: {camera}")
            continue
        samples_dir = cfg.outputs_dir / "labeling" / "sleap" / f"{session_id}" / camera

        if CLIP_START_S is not None and CLIP_DURATION_S is not None:
            t0 = float(CLIP_START_S)
            t1 = t0 + float(CLIP_DURATION_S)
            if "t" not in ft_cam.columns:
                print("No timestamps 't' for clip selection; falling back to sampling.")
                frame_idx = sample_frame_indices(ft_cam, n_samples=N_SAMPLES)
            else:
                ft_clip = ft_cam[(ft_cam["t"] >= t0) & (ft_cam["t"] <= t1)]
                if ft_clip.empty:
                    print(f"No frames found in clip window t=[{t0}, {t1}] for camera: {camera}")
                    frame_idx = sample_frame_indices(ft_cam, n_samples=N_SAMPLES)
                else:
                    frame_idx = ft_clip["frame_idx"].astype(int).to_numpy()
        else:
            frame_idx = sample_frame_indices(ft_cam, n_samples=N_SAMPLES)

        mode = LABELING_MODE.lower()
        if mode in {"mp4", "video"}:
            export_labeling_video(
                Path(video_path),
                frame_idx,
                samples_dir,
                ft_cam,
                session_id,
                camera,
                label_fps=LABEL_FPS,
                write_pngs=WRITE_PNGS,
            )
        elif mode in {"png", "frames"}:
            export_labeling_frames(Path(video_path), frame_idx, samples_dir, ft_cam, session_id, camera)
        else:
            raise ValueError(f"Unknown LABELING_MODE: {LABELING_MODE}")

        print("Labeling outputs:", samples_dir)

    print("Pose project:", project_dir)


Labeling outputs: /Users/muh/projects/vbn-analysis/outputs/labeling/sleap/1043752325/eye
Labeling outputs: /Users/muh/projects/vbn-analysis/outputs/labeling/sleap/1043752325/face
Labeling outputs: /Users/muh/projects/vbn-analysis/outputs/labeling/sleap/1043752325/side
Pose project: /Users/muh/projects/vbn-analysis/pose_projects/session_1043752325/sleap
